# Семинар 1. Gymnasium, бандиты, метод Cross-Entropy

План:

1. Интерфейс Gymnasium на примере `FrozenLake-v1`
2. Политика как таблица: пишем маршрут по льду руками, что ломается на скользком льду
3. Своя среда многорукого бандита, агенты ε-greedy и UCB1, сравнение regret
4. Метод Cross-Entropy на Frozen Lake
5. Что дальше: PyTorch-разминка и домашнее задание

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

rng = np.random.default_rng(42)

## 1. Интерфейс Gymnasium

Любая среда в Gymnasium следует единому интерфейсу:

* `env.reset(seed=...)` -> `(observation, info)` — сбросить среду в начальное состояние
* `env.step(action)` -> `(observation, reward, terminated, truncated, info)` — сделать шаг
* `env.observation_space`, `env.action_space` — описание пространств состояний/действий

`terminated` — эпизод закончился естественным образом (дошли до цели или упали в прорубь),
`truncated` — эпизод прерван искусственно (по лимиту шагов).

Начнём с **FrozenLake**: агент ходит по замёрзшему озеру 4×4 от старта `S` к цели `G`,
в клетках `H` — проруби. Награда +1 только за достижение цели.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
print("observation_space:", env.observation_space)
print("action_space:", env.action_space)

obs, info = env.reset(seed=0)
print("начальное состояние:", obs)

for step in range(5):
    action = env.action_space.sample()  # случайное действие
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step={step} action={action} obs={obs} reward={reward} terminated={terminated}")
    if terminated or truncated:
        obs, info = env.reset()

env.close()

Состояние здесь — номер клетки от 0 до 15 (по строкам, слева направо).
Действия: `0` — влево, `1` — вниз, `2` — вправо, `3` — вверх.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
obs, _ = env.reset(seed=0)
desc = env.unwrapped.desc.astype(str)
print("карта озера (S — старт, F — лёд, H — прорубь, G — цель):")
for row in desc:
    print("  ", " ".join(row))
print()
print("номера состояний:")
print(np.arange(16).reshape(4, 4))

plt.imshow(env.render())
plt.axis("off")
plt.title("FrozenLake-v1: агент в состоянии 0")
plt.show()
env.close()

## 2. Политика как таблица

Когда состояний всего 16, политику можно записать **таблицей**: для каждой клетки одно действие.
Именно такую таблицу в разделе 4 будет учить алгоритм. Сначала напишем её руками.

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

policy = {0: DOWN, 4: DOWN, 8: RIGHT, 9: DOWN, 13: RIGHT, 14: RIGHT}


def table_policy(obs):
    return policy.get(int(obs), LEFT)


def run_episode(env, policy_fn, seed=None, verbose=False):
    obs, _ = env.reset(seed=seed)
    total, t = 0.0, 0
    while True:
        action = policy_fn(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        if verbose:
            print(f"t={t}: состояние {obs} -> действие {action} -> состояние {next_obs}, награда {reward}")
        total += reward
        obs = next_obs
        t += 1
        if terminated or truncated:
            return total


def success_rate(env, policy_fn, n_episodes=500):
    return np.mean([run_episode(env, policy_fn, seed=ep) > 0 for ep in range(n_episodes)])


env = gym.make("FrozenLake-v1", is_slippery=False)
print("суммарная награда за эпизод:", run_episode(env, table_policy, seed=0, verbose=True))
env.close()

for slippery in [False, True]:
    env = gym.make("FrozenLake-v1", is_slippery=slippery)
    print(f"is_slippery={slippery!s:5}: таблица {success_rate(env, table_policy):.1%}, "
          f"случайная политика {success_rate(env, lambda o: int(rng.integers(4))):.1%}")
    env.close()

На скользком льду агент идёт туда, куда хотел, с вероятностью 1/3, иначе его сносит вбок. Маршрут,
идеальный на гладком льду, доходит до цели в единицах процентов случаев: его сносит в проруби,
а таблица не знает, что делать в клетках, куда агент «не собирался».

Попробуйте вживую: дополните таблицу для всех 16 клеток и найдите более надёжный маршрут.
Подсказка: на скользком льду выгодно «идти в стену», потому что стена не пускает, а снос работает
в нужную сторону. Оптимальную таблицу получим автоматически на неделе 3.

In [ ]:
policy_slippery = dict(policy)
# policy_slippery[1] = ...
env = gym.make("FrozenLake-v1", is_slippery=True)
print(f"улучшенная таблица на скользком льду: {success_rate(env, lambda o: policy_slippery.get(int(o), LEFT)):.1%}")
env.close()

## 3. Своя среда многорукого бандита

Напишем среду в стиле Gymnasium, но без наследования от `gym.Env` — минимальный класс.

In [ ]:
class BernoulliBanditEnv:
    # K рук, у каждой руки k своя вероятность успеха p_k (награда 0 или 1)

    def __init__(self, probs, rng=None):
        self.probs = np.array(probs, dtype=float)
        self.n_arms = len(probs)
        self.rng = rng or np.random.default_rng()

    def pull(self, arm: int) -> float:
        return float(self.rng.random() < self.probs[arm])

    @property
    def optimal_mean(self) -> float:
        return self.probs.max()


bandit = BernoulliBanditEnv(probs=[0.1, 0.5, 0.3, 0.55, 0.45], rng=rng)
print("оптимальная рука:", bandit.probs.argmax(), "с вероятностью", bandit.optimal_mean)

### Агенты

Два агента с общим интерфейсом: `select_arm()` и `update(arm, reward)`. Оценку $\hat Q(a)$ обновляем
инкрементально: $\hat Q \leftarrow \hat Q + \frac{1}{N}(r - \hat Q)$, это то же самое, что среднее по всем наградам руки.

In [ ]:
class EpsilonGreedyAgent:
    def __init__(self, n_arms, epsilon=0.1, rng=None):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.rng = rng or np.random.default_rng()

    def select_arm(self) -> int:
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.n_arms))
        return int(np.argmax(self.Q))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


class UCB1Agent:
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.t = 0

    def select_arm(self) -> int:
        self.t += 1
        untried = np.where(self.N == 0)[0]      # каждую руку пробуем хотя бы раз
        if len(untried) > 0:
            return int(untried[0])
        bonus = self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(self.Q + bonus))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

### Сравнение по regret

Запустим каждого агента на одном и том же бандите и посчитаем накопленный regret, усреднив по сидам.

In [ ]:
def run_agent(agent_factory, bandit_probs, n_steps=2000, n_seeds=20):
    cum_regrets = np.zeros((n_seeds, n_steps))
    for seed in range(n_seeds):
        local_rng = np.random.default_rng(seed)
        bandit = BernoulliBanditEnv(bandit_probs, rng=local_rng)
        agent = agent_factory(local_rng)
        regret = np.zeros(n_steps)
        for t in range(n_steps):
            arm = agent.select_arm()
            reward = bandit.pull(arm)
            agent.update(arm, reward)
            regret[t] = bandit.optimal_mean - bandit.probs[arm]
        cum_regrets[seed] = np.cumsum(regret)
    return cum_regrets.mean(axis=0)


bandit_probs = [0.1, 0.5, 0.3, 0.55, 0.45]
n_arms = len(bandit_probs)

agents = {
    "greedy (eps=0)": lambda rng: EpsilonGreedyAgent(n_arms, epsilon=0.0, rng=rng),
    "epsilon-greedy (eps=0.1)": lambda rng: EpsilonGreedyAgent(n_arms, epsilon=0.1, rng=rng),
    "epsilon-greedy (eps=0.01)": lambda rng: EpsilonGreedyAgent(n_arms, epsilon=0.01, rng=rng),
    "UCB1 (c=2)": lambda rng: UCB1Agent(n_arms, c=2.0),
}

plt.figure(figsize=(7, 5))
for name, factory in agents.items():
    plt.plot(run_agent(factory, bandit_probs), label=name)
plt.xlabel("шаг t")
plt.ylabel("средний накопленный regret")
plt.title("Сравнение стратегий на 5-руком Bernoulli-бандите")
plt.legend()
plt.show()

Обратите внимание на форму кривых: у ε-greedy regret растёт **линейно** даже после нахождения лучшей руки
(постоянный шанс ε продолжать исследовать), у UCB1 рост **замедляется**: он реже трогает руки, в которых уже уверен.
Жадный агент без exploration иногда залипает на плохой руке, и тогда regret растёт быстрее всех.

## 4. Метод Cross-Entropy на Frozen Lake

Реализуем алгоритм из лекции. Политика — таблица `n_states × n_actions` с вероятностями действий.

* `get_action(state)` — сэмплирует действие из `policy[state]`;
* `update_policy(elite_sessions)` — считает частоты пар (состояние, действие) в элитных сессиях.

In [ ]:
class CEMAgent:
    def __init__(self, n_states, n_actions, rng=None):
        self.n_states, self.n_actions = n_states, n_actions
        self.policy = np.ones((n_states, n_actions)) / n_actions
        self.rng = rng or np.random.default_rng()

    def get_action(self, state) -> int:
        return int(self.rng.choice(self.n_actions, p=self.policy[state]))

    def update_policy(self, elite_sessions, laplace=0.0, mix=1.0):
        counts = np.full((self.n_states, self.n_actions), laplace)
        for session in elite_sessions:
            for s, a in zip(session["states"], session["actions"]):
                counts[s, a] += 1
        new_policy = self.policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        self.policy = mix * new_policy + (1 - mix) * self.policy


def get_session(env, agent, max_steps=100):
    states, actions, total = [], [], 0.0
    obs, _ = env.reset(seed=int(agent.rng.integers(1_000_000)))
    for _ in range(max_steps):
        a = agent.get_action(obs)
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return {"states": states, "actions": actions, "total": total}


def train_cem(env, n_iter=20, n_sessions=100, q=0.7, laplace=0.0, mix=1.0, seed=0):
    agent = CEMAgent(env.observation_space.n, env.action_space.n, rng=np.random.default_rng(seed))
    history = []
    for it in range(n_iter):
        sessions = [get_session(env, agent) for _ in range(n_sessions)]
        totals = np.array([s["total"] for s in sessions])
        history.append(totals.mean())
        threshold = np.quantile(totals, q)
        elite = [s for s in sessions if s["total"] >= threshold and s["total"] > 0]
        agent.update_policy(elite, laplace=laplace, mix=mix)
    return agent, history


env = gym.make("FrozenLake-v1", is_slippery=False)
agent_plain, hist_plain = train_cem(env)
agent_smooth, hist_smooth = train_cem(env, laplace=0.5, mix=0.5)

plt.plot(hist_plain, marker="o", label="без сглаживания")
plt.plot(hist_smooth, marker="o", label="Лаплас λ=0.5, смесь 0.5")
plt.xlabel("итерация"); plt.ylabel("средний return (доля успехов)")
plt.title("Cross-Entropy на FrozenLake 4x4 без скольжения")
plt.legend(); plt.show()

arrows = "←↓→↑"
print("выученная политика:")
for row in range(4):
    print("  " + " ".join(arrows[int(np.argmax(agent_smooth.policy[row * 4 + col]))] for col in range(4)))
print(f"доля успехов выученной политики: {success_rate(env, lambda o: int(np.argmax(agent_smooth.policy[o]))):.1%}")
env.close()

Вопросы для обсуждения и экспериментов вживую:

* Что будет на скользком льду (`is_slippery=True`)? Запустите и посмотрите на кривую. Почему элитные траектории там обманывают?
* Как влияют `q` (доля элиты) и `n_sessions`? Попробуйте `q=0.9` и `n_sessions=20`.
* Сравните выученную таблицу с той, что вы написали руками в разделе 2.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
agent_slip, hist_slip = train_cem(env, n_iter=30, n_sessions=200, laplace=0.5, mix=0.5)
plt.plot(hist_slip, marker="o")
plt.xlabel("итерация"); plt.ylabel("средний return")
plt.title("Cross-Entropy на скользком FrozenLake")
plt.show()
print(f"доля успехов: {success_rate(env, lambda o: int(np.argmax(agent_slip.policy[o]))):.1%}")
env.close()

## Что дальше

* **Мини-семинар по PyTorch** (`pytorch_intro.ipynb`): тензоры, autograd, `nn.Module`, цикл обучения и первый
  «агент на нейросети» для CartPole.
* **Домашнее задание** (`../homework/homework.ipynb`): бандиты (ε-greedy с расписанием, UCB1, сравнение на разных
  конфигурациях), теория (return, вывод уравнения Беллмана, марковская цепь, MDP на бумаге), Cross-Entropy на Frozen Lake 8×8.
* **Неделя 2**: ключевые понятия подробнее и построение собственной среды в Gymnasium.